In [1]:
# --- 1. Instalación ---
# CORRECCIÓN: Instalamos desde PyPI (oficial) para evitar el error de "Username...".
!pip install "unsloth[llama-3-8b]"
!pip install "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.1" "trl>=0.8.3" "peft>=0.10.0" "bitsandbytes>=0.43.1"

# --- Imports (de vuelta al inicio) ---
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset
import os

# Asegúrate de que Kaggle/PyTorch pueda ver ambas GPUs
os.environ["WANDB_DISABLED"] = "true" # Desactiva W&B si no lo vas a usar

# --- 2. Cargar Modelo y Tokenizador ---
max_seq_length = 4096 
dtype = None 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # Eliminamos 'device_map'. El modelo se cargará en la GPU 
    # por defecto (cuda:0)
)

# --- 3. Configuración de LoRA (PEFT) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = True,
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# --- 4. Cargar el Conjunto de Datos ---
data_path = "/kaggle/input/result2/result.txt"
dataset = load_dataset("text", data_files={"train": data_path})

# --- 5. Configurar el Entrenamiento ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text", 
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, 
    packing = True, 

    args = TrainingArguments(
        # 'per_device_train_batch_size' ahora se aplica solo a 1 GPU
        per_device_train_batch_size = 2, 
        gradient_accumulation_steps = 4, 
        warmup_steps = 10,
        
        # --- Configuración de Pasos ---
        max_steps = 15500,
        # -------------------------------

        learning_rate = 1e-4, 
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        
        logging_steps = 10, 
        optim = "adamw_8bit", 
        weight_decay = 0.01,
        lr_scheduler_type = "linear", 
        seed = 42,
        
        output_dir = "/kaggle/working/outputs", 
        remove_unused_columns = True,
        report_to = "none",
    ),
)

# --- 6. Iniciar el Entrenamiento ---
print("Iniciando entrenamiento en 1 GPU...")
trainer.train(resume_from_checkpoint=True)
print("Entrenamiento completado.")

# --- 7. Guardar el Modelo ---
output_model_path = "/kaggle/working/llama3-8b-4chan-unsloth"
trainer.save_model(output_model_path)
tokenizer.save_pretrained(output_model_path)
print(f"Modelo guardado en {output_model_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-19 15:06:27.171456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760886387.195815     490 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760886387.203829     490 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.6: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.10.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Iniciando entrenamiento en 1 GPU...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,106,075 | Num Epochs = 1 | Total steps = 15,500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
13010,3.760600
13020,3.620300
13030,3.600100
13040,3.545700
13050,3.567000
13060,3.575500
13070,3.809600
13080,3.519000
13090,3.683400
13100,3.602400


Unsloth: Will smartly offload gradients to save VRAM!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Entrenamiento completado.
Modelo guardado en /kaggle/working/llama3-8b-4chan-unsloth


In [1]:
!pip install "unsloth[llama-3-8b]"
!pip install "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.1" "trl>=0.8.3" "peft>=0.10.0" "bitsandbytes>=0.43.1"
import torch
from unsloth import FastLanguageModel
import sys
# ¡Nuevas importaciones!
from transformers import StoppingCriteria, StoppingCriteriaList

# --- 0. Definir el Criterio de Parada ---
# Esto es una clase que le dice a model.generate() que pare
# si ve la cadena "---"

# Nota: El tokenizer debe estar cargado antes de esto, así que 
# movemos la carga del modelo al principio.

# --- 1. Cargar tu Modelo Entrenado ---
model_path = "llama3-8b-4chan-unsloth"

print("Cargando modelo... (Esto puede tardar unos segundos)")
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 4096,
        dtype = None,
        load_in_4bit = True,
    )
except Exception as e:
    print(f"Error al cargar el modelo. Asegúrate de que la carpeta '{model_path}' esté en el mismo directorio.")
    print(f"Error: {e}")
    sys.exit()

FastLanguageModel.for_inference(model)
print("Modelo cargado exitosamente.")

# --- Definir el Criterio de Parada (continuación) ---
# Tokenizamos la "palabra de parada"
stop_sequence = "---"
stop_token_ids = tokenizer.encode(stop_sequence, add_special_tokens=False)
stop_token_ids = torch.tensor(stop_token_ids).to("cuda")

class StopOnTokens(StoppingCriteria):
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Revisa si los *últimos* N tokens coinciden con la secuencia de parada
        if input_ids.shape[1] < len(stop_token_ids):
            return False
        
        last_tokens = input_ids[0, -len(stop_token_ids):]
        # Si los últimos tokens son "---", devolvemos True
        if torch.equal(last_tokens, stop_token_ids):
            return True # ¡Parar la generación!
        return False

# Crear la lista de criterios para pasarla al generador
stopping_criteria = StoppingCriteriaList([StopOnTokens()])


# --- 2. Personalidad (In-Context Learning) ---
PERSONALITY_CONTEXT = """--- 99999701
This faggot >>99999702 posts like a fucking bot, I swear.
--- 99999702
>>99999701
The fuck are you talking about, schizo. I'll type however I want, I'm a fucking person.
--- 99999703
>>99999702
idk, you sound like an AI. Prove it.
--- 99999704
>>99999703
"you sound like an AI"... just go to bed, kid. Of course I'm a person, you goddamn retard. I don't have to prove shit to you.
--- 99999705
>>99999704
Based. Fuck 'em.
"""

# --- 3. Formato del Prompt ---
prompt_template = PERSONALITY_CONTEXT + """
--- 99999901
{user_prompt}
--- 99999902
>>99999901
"""

# --- 4. Bucle del Chat ---
print("\n=== Chatbot 4chan-GPT Iniciado ===")
print("Escribe tu mensaje. Escribe 'salir' para terminar.")

while True:
    pregunta_usuario = input("\nTú: ")
    if pregunta_usuario.lower() == 'salir':
        break
    
    prompt = prompt_template.format(user_prompt=pregunta_usuario)
    inputs = tokenizer([prompt], return_tensors="pt", truncation=True).to("cuda")

    # --- 5. Generar la respuesta (Con Optimización) ---
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.eos_token_id,
        
        do_sample = True,      
        temperature = 0.7,   
        top_p = 0.9,
        
        # --- ¡AQUÍ ESTÁ LA MAGIA! ---
        # 1. Penaliza las repeticiones (evita alucinaciones/bucles)
        repetition_penalty = 1.15,
        
        # 2. Le dice al modelo que pare cuando vea "---"
        stopping_criteria = stopping_criteria,
    )

    # --- 6. Decodificar y Limpiar la Respuesta ---
    response_full = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Limpiamos el prompt
    response_only = response_full.split(prompt, 1)[-1].strip()
    
    # Limpiamos el "---" si se coló al final.
    response_only = response_only.split("---")[0].strip()

    print(f"\nModelo: {response_only}")

print("Chatbot finalizado.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 29.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.2/269.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 15.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 2.7 MB/s eta 0:00:00:00:0100

ImportError: Unsloth: Please install unsloth_zoo via `pip install unsloth_zoo`